# 08 | Burkina Faso: crop classification and validation
**Ibrahim Kabore · WorldCereal extension · Koumbia**

Notebook 07 produced two 128-dimensional representations of the same 1,525 field observations. I now compare whether using available radar measurements helps a downstream classifier distinguish six crop types.

The pretrained encoder remains frozen. The only supervised models trained here are Random Forest and histogram gradient boosting. A majority-class baseline shows how much improvement is achieved over always predicting the most frequent training class.

**Evaluation plan:** train on 2017, 2018 and 2020; select the primary model using 2021 validation macro F1; evaluate the fixed candidates on 2022 only after selection. Separately, use five-fold spatial block validation on the training years. The temporal and spatial experiments answer different questions. CPU runtime is sufficient.

## 1. Connect Drive and import the analysis libraries
No GPU or WorldCereal installation is required here because the embeddings are already saved.

In [ ]:
from google.colab import drive
from pathlib import Path
import json, time, hashlib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import sklearn, joblib
from sklearn.base import clone
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.metrics import accuracy_score, balanced_accuracy_score, f1_score, classification_report, ConfusionMatrixDisplay
from sklearn.model_selection import StratifiedGroupKFold
from scipy.spatial import cKDTree
from pyproj import Transformer

drive.mount("/content/drive")
PROJECT = Path("/content/drive/MyDrive/ESA_EO_Projects/WorldCereal_Burkina_Faso")
OUT = PROJECT / "outputs"
MODELS = PROJECT / "models"
RESULTS = OUT / "classification"
RESULTS.mkdir(parents=True,exist_ok=True)
MODELS.mkdir(parents=True,exist_ok=True)
SEED=42
sns.set_theme(style="whitegrid")
print("scikit-learn:",sklearn.__version__)

## 2. Verify matched inputs and provenance
Both embedding tables must represent the same field observations in the same order. Crop labels, coordinates, years and spatial groups are metadata, never extra classifier features. Check both manifests to prevent mixing model versions or seasonal definitions.

In [ ]:
KEYS=["ref_id","sample_id"]
CONFIGS=["optical","multisensor"]
F=[f"presto_ft_{i}" for i in range(128)]
META=KEYS+["label","label_year","split","spatial_group","lon","lat","ewoc_code","optical_months","radar_months"]
EXPECTED_SHA="e16dce35b55b368c1553dd6c6dfe26f39d27818f24a9d710e2922cd4f00fed88"
tables={}
manifests={}
for name in CONFIGS:
    d=pd.read_parquet(OUT/f"reference_embeddings_{name}.parquet")
    m=json.loads((MODELS/f"embedding_manifest_{name}.json").read_text())
    assert set(F+META)<=set(d.columns)
    assert not d.duplicated(KEYS).any()
    assert d[META].notna().all().all() and np.isfinite(d[F]).all().all()
    assert m["features"]==F and m["checkpoint_sha256"]==EXPECTED_SHA
    assert m["pool_months"]==[5,6,7,8,9,10] and m["context"]=="Jan-Dec"
    assert m["mask_b8a"] is True and m["use_radar"]==(name=="multisensor")
    assert len(d)==m["sample_count"]
    tables[name]=d.sort_values(KEYS).reset_index(drop=True)
    manifests[name]=m
pd.testing.assert_frame_equal(tables["optical"][META],tables["multisensor"][META])
meta=tables["optical"][META].copy()
CLASSES=sorted(meta.label.unique())
assert CLASSES==["Cotton","Cowpea","Groundnut","Maize","Sorghum","Soybean"]
assert meta.split.eq(meta.label_year.map({2017:"train",2018:"train",2020:"train",2021:"validation",2022:"test"})).all()
display(pd.crosstab(meta.label,meta.split,margins=True))
print("Matched observations:",len(meta),"| spatial blocks:",meta.spatial_group.nunique())

## 3. Inspect class imbalance and spatial proximity
Accuracy can favour abundant crops. Balanced accuracy gives equal weight to class recall; macro F1 gives equal weight to the harmonic mean of precision and recall for each crop. Measure distances on the actual extracted points, rather than reusing the earlier 2,964-point analysis.

In [ ]:
counts=pd.crosstab(meta.label,meta.split)
assert (counts>0).all().all()
counts.plot.bar(figsize=(10,4),rot=25)
plt.title("Crop representation by temporal split")
plt.ylabel("Samples")
plt.tight_layout()
plt.savefig(RESULTS/"split_distribution.png",dpi=160)
plt.show()
transformer=Transformer.from_crs(4326,32630,always_xy=True)
x,y=transformer.transform(meta.lon.to_numpy(),meta.lat.to_numpy())
xy=np.column_stack([x,y])
train_idx=np.flatnonzero(meta.split.eq("train"))
val_idx=np.flatnonzero(meta.split.eq("validation"))
test_idx=np.flatnonzero(meta.split.eq("test"))
tree=cKDTree(xy[train_idx])
for name,idx in [("validation",val_idx),("test",test_idx)]:
    dist,_=tree.query(xy[idx])
    print(name, "median nearest training distance (m):",round(np.median(dist),1))
    print({f"within_{t}m_pct":round(100*(dist<=t).mean(),1) for t in [10,50,100,500]})

## 4. Define fixed classifiers and metrics
The two feature configurations are compared with identical model settings. Random Forest uses balanced class weights to reduce the influence of class imbalance. The gradient boosting model is a second nonlinear classifier. Hyperparameters are fixed before looking at validation/test outcomes.

In [ ]:
def candidates():
    return {
        "majority":DummyClassifier(strategy="most_frequent"),
        "random_forest":RandomForestClassifier(n_estimators=150,min_samples_leaf=2,
            class_weight="balanced_subsample",n_jobs=2,random_state=SEED),
        "hist_gradient_boosting":HistGradientBoostingClassifier(max_iter=100,max_leaf_nodes=31,
            learning_rate=0.1,random_state=SEED)
    }
def metrics(y,p):
    return {"accuracy":accuracy_score(y,p),
            "balanced_accuracy":balanced_accuracy_score(y,p),
            "macro_f1":f1_score(y,p,labels=CLASSES,average="macro",zero_division=0)}
Y=meta.label.to_numpy()
X={name:tables[name][F].to_numpy(dtype=np.float32) for name in CONFIGS}
fitted={}
rows=[]
for name in CONFIGS:
    for model_name,model in candidates().items():
        start=time.perf_counter()
        model.fit(X[name][train_idx],Y[train_idx])
        elapsed=time.perf_counter()-start
        pred=model.predict(X[name][val_idx])
        rows.append({"configuration":name,"model":model_name,"seconds":elapsed,**metrics(Y[val_idx],pred)})
        fitted[(name,model_name)]=model
validation=pd.DataFrame(rows).sort_values(["macro_f1","balanced_accuracy","configuration","model"],ascending=[False,False,True,True])
display(validation.round(4))
validation.to_csv(RESULTS/"validation_metrics.csv",index=False)

## 5. Lock the primary model before testing
Choose the highest validation macro F1 among the real classifiers, with balanced accuracy and deterministic alphabetical tie-breaks. Save the decision before evaluating 2022. The validation year contains no radar observations, so this selection targets performance under its observed sensor conditions; the 2022 test assesses a different condition.

In [ ]:
eligible=validation[validation.model.ne("majority")]
winner=eligible.iloc[0]
PRIMARY=(winner.configuration,winner.model)
decision={"configuration":PRIMARY[0],"model":PRIMARY[1],
          "selection_metric":"validation_macro_f1",
          "validation_macro_f1":float(winner.macro_f1),
          "fit_years":[2017,2018,2020],"validation_year":2021,"test_year":2022}
(RESULTS/"model_selection.json").write_text(json.dumps(decision,indent=2))
print("Selected on validation only:",PRIMARY)
print("Validation macro F1:",round(winner.macro_f1,4))

## 6. Evaluate the fixed models on 2022
All candidate results are reported for the planned comparison. Do not change the selected model or tune hyperparameters based on the test table. These scores measure year transfer within the reference region, not independent spatial map accuracy.

In [ ]:
test_rows=[]
predictions=meta.iloc[test_idx][META].reset_index(drop=True).copy()
for (name,model_name),model in fitted.items():
    pred=model.predict(X[name][test_idx])
    predictions[f"{name}_{model_name}"]=pred
    test_rows.append({"configuration":name,"model":model_name,**metrics(Y[test_idx],pred)})
test_results=pd.DataFrame(test_rows)
display(test_results.round(4))
test_results.to_csv(RESULTS/"test_metrics.csv",index=False)
predictions.to_csv(RESULTS/"test_predictions.csv",index=False)
for split_name,table in [("Validation",validation),("Test",test_results)]:
    fig,ax=plt.subplots(figsize=(10,4))
    table.pivot(index="model",columns="configuration",values="macro_f1").plot.bar(ax=ax,rot=0)
    ax.set(title=f"{split_name} | Macro F1",ylabel="Macro F1",ylim=(0,1))
    fig.tight_layout()
    fig.savefig(RESULTS/f"{split_name.lower()}_macro_f1.png",dpi=160)
    plt.show()
print("Preselected model:",PRIMARY)
display(test_results.loc[(test_results.configuration==PRIMARY[0])&(test_results.model==PRIMARY[1])])

## 7. Examine class-level errors
Rows of the normalized confusion matrix represent true crop classes, and columns represent predictions. Report class support alongside precision, recall and F1. Small test classes have greater sampling uncertainty.

In [ ]:
for name in CONFIGS:
    model=fitted[(name,"random_forest")]
    pred=model.predict(X[name][test_idx])
    report=pd.DataFrame(classification_report(Y[test_idx],pred,labels=CLASSES,output_dict=True,zero_division=0)).T
    print(name,"| Random Forest")
    display(report.round(3))
    report.to_csv(RESULTS/f"classification_report_{name}_random_forest.csv")
    fig,ax=plt.subplots(figsize=(7,6))
    ConfusionMatrixDisplay.from_predictions(Y[test_idx],pred,labels=CLASSES,normalize="true",
        cmap="Blues",values_format=".2f",xticks_rotation=45,ax=ax,colorbar=False)
    ax.set_title(f"2022 test | {name} Random Forest")
    fig.tight_layout()
    fig.savefig(RESULTS/f"confusion_{name}_random_forest.png",dpi=160)
    plt.show()
model=fitted[PRIMARY]
pred=model.predict(X[PRIMARY[0]][test_idx])
report=pd.DataFrame(classification_report(Y[test_idx],pred,labels=CLASSES,output_dict=True,zero_division=0)).T
print("Preselected model:",PRIMARY)
display(report.round(3))
report.to_csv(RESULTS/"selected_model_report.csv")

## 8. Spatial block cross-validation (training years only)
Use identical 2 km block assignments across both feature sets. Withhold complete spatial groups, then remove training points within 100 m of a held-out point as a sensitivity analysis against adjacent-block proximity. Report fold sizes and missing classes. This is a secondary development-set experiment, not another test on 2022.

In [ ]:
groups=meta.spatial_group.to_numpy()
cv=StratifiedGroupKFold(n_splits=5,shuffle=True,random_state=SEED)
cv_rows=[]
fold_rows=[]
for fold,(tr,te) in enumerate(cv.split(X["optical"][train_idx],Y[train_idx],groups[train_idx]),1):
    fit_idx=train_idx[tr]
    held_idx=train_idx[te]
    assert set(groups[fit_idx]).isdisjoint(groups[held_idx])
    distances,_=cKDTree(xy[held_idx]).query(xy[fit_idx])
    for buffer_m in [0,100]:
        use=fit_idx if buffer_m==0 else fit_idx[distances>buffer_m]
        missing_train=sorted(set(CLASSES)-set(Y[use]))
        missing_test=sorted(set(CLASSES)-set(Y[held_idx]))
        fold_rows.append({"fold":fold,"buffer_m":buffer_m,"train":len(use),"held_out":len(held_idx),
                          "train_groups":len(set(groups[use])),"held_groups":len(set(groups[held_idx])),
                          "missing_train":",".join(missing_train),"missing_test":",".join(missing_test)})
        if missing_train or missing_test:
            print("Skipping incomplete class coverage:",fold,buffer_m,missing_train,missing_test)
            continue
        for name in CONFIGS:
            for model_name in ["random_forest","hist_gradient_boosting"]:
                model=clone(candidates()[model_name])
                model.fit(X[name][use],Y[use])
                pred=model.predict(X[name][held_idx])
                cv_rows.append({"fold":fold,"buffer_m":buffer_m,"configuration":name,"model":model_name,**metrics(Y[held_idx],pred)})
fold_info=pd.DataFrame(fold_rows)
display(fold_info)
fold_info.to_csv(RESULTS/"spatial_fold_audit.csv",index=False)
spatial_scores=pd.DataFrame(cv_rows)
spatial_scores.to_csv(RESULTS/"spatial_cv_metrics.csv",index=False)
if len(spatial_scores):
    summary=spatial_scores.groupby(["buffer_m","configuration","model"])[["accuracy","balanced_accuracy","macro_f1"]].agg(["count","mean","std"])
    display(summary.round(4))
    summary.to_csv(RESULTS/"spatial_cv_summary.csv")
else:
    print("No eligible spatial folds; revise group design without using 2022 test outcomes.")

## 9. Save the model and its inference contract
Save the exact validation-selected classifier fitted on the original training years, not a model refitted on the test data. Preserve the feature order and encoder fingerprint for spatial mapping. Maximum class probability is an uncalibrated confidence score. Only load locally generated joblib files later.

In [ ]:
name,model_name=PRIMARY
selected=fitted[PRIMARY]
path=MODELS/"selected_classifier.joblib"
joblib.dump(selected,path)
manifest={**manifests[name],**decision,"sklearn_version":sklearn.__version__,
          "classifier_file":path.name,"classifier_sha256":hashlib.sha256(path.read_bytes()).hexdigest(),
          "classes":selected.classes_.tolist(),"crop_codes":{str(i+1):c for i,c in enumerate(selected.classes_)},
          "fit_sample_count":len(train_idx),"validation_sample_count":len(val_idx),"test_sample_count":len(test_idx)}
(MODELS/"classifier_manifest.json").write_text(json.dumps(manifest,indent=2))
print("Saved:",path)
print("Selected model:",PRIMARY)
print("Crop codes:",manifest["crop_codes"])
print("Results:",RESULTS)

## 10. Interpretation and next steps
Compare optical and multisensor performance on identical samples. Radar is absent in 2021 but available in 2022, so differences may depend on sensor availability and year-specific conditions. The spatial evaluation addresses a different question from temporal transfer, and nearby samples can remain correlated even with 2 km grouping. Potential overlap with foundation-model pretraining is unknown. These six crop classes do not cover every possible land-cover or crop category in Koumbia; spatial mapping will require a cropland mask and careful handling of unsupported crops.

Notebook 09 will prepare a compatible 2022 spatial cube, use the selected inference configuration, and export crop and confidence maps. No spatial accuracy is established by this notebook alone.